# 05 — Assemble the Track-2 stance records

**Purpose.** Combine the five production shards (notebook 03, prompt
`stance-extraction-0.3-dev`, array job 11261709) into the stage's two
corpus-level tables, with the verification that decides whether the
production run is usable:

- `stance_turns.parquet` — one row per legislator turn (coverage, errors,
  merge counts);
- `stance_records.parquet` — one row per stance record, the atomic unit of
  Track 2: (speaker, evidence sentence indices, neutral-noun-phrase topic,
  stance ∈ {support, concern}).

**Gate.** Coverage must equal the full non-empty worklist from notebook 01,
keys must be unique, and the corpus-level error and lint rates must sit in
the bands measured on the dev split. Out-of-band numbers stop the pipeline
here — the downstream taxonomy never sees an unverified corpus.

**Inputs.** `interim/topics/track2/shards/records_{00..04}.parquet`,
`interim/topics/track2/member_turns.parquet` (01).

**Outputs.** `interim/topics/track2/stance_turns.parquet`,
`interim/topics/track2/stance_records.parquet`.

## Step 1 — Combine shards and check coverage

The five shards partition the stored turn order (`iloc[shard::5]`), so
their union must reproduce the worklist exactly: every non-empty turn
once, none twice.

In [1]:
import json
import sys

sys.path.insert(0, "../lib")

import pandas as pd

import stance
from paths import topics_run_dir

TRACK2 = topics_run_dir("track2")

turns = pd.concat([pd.read_parquet(TRACK2 / "shards" / f"records_{s:02d}.parquet")
                   .assign(shard=s) for s in range(5)], ignore_index=True)
work = pd.read_parquet(TRACK2 / "member_turns.parquet",
                       columns=["packageId", "turn_idx", "n_sentences"])
work = work[work["n_sentences"] > 0]

assert not turns.duplicated(["packageId", "turn_idx"]).any(), "turn done twice"
missing = (set(zip(work["packageId"], work["turn_idx"]))
           - set(zip(turns["packageId"], turns["turn_idx"])))
assert not missing, f"{len(missing)} worklist turns missing"
assert len(turns) == len(work), "extra turns beyond the worklist"
print(f"turns: {len(turns):,} — exactly the non-empty worklist, no duplicates")
print(turns.groupby("shard").size().to_string())

turns: 220,160 — exactly the non-empty worklist, no duplicates
shard
0    44020
1    44034
2    44046
3    44024
4    44036


## Step 2 — One row per stance record

`records_json` holds the machine-validated, adjacency-merged records from
notebook 03. Exploding preserves the full evidence chain: each record keeps
its turn key, so record → sentence indices → `member_turns.parquet` text →
transcript remains a lookup, never a search.

In [2]:
rows = []
for t in turns.itertuples():
    for i, r in enumerate(json.loads(t.records_json)):
        rows.append({
            "packageId": t.packageId, "turn_idx": t.turn_idx,
            "year": t.year, "chamber": t.chamber,
            "bioGuideId": t.bioGuideId, "surname": t.surname,
            "honorific": t.honorific, "is_chair": t.is_chair,
            "record_idx": i, "sentences": r["sentences"],
            "topic": r["topic"], "stance": r["stance"],
            "topic_lint": r["topic_lint"],
        })
records = pd.DataFrame(rows)

turns.drop(columns=["records_json"]).to_parquet(
    TRACK2 / "stance_turns.parquet", compression="zstd", index=False)
records.to_parquet(TRACK2 / "stance_records.parquet", compression="zstd",
                   index=False)
print(f"{len(records):,} stance records -> stance_records.parquet")
print(f"{len(turns):,} turns          -> stance_turns.parquet")

32,385 stance records -> stance_records.parquet
220,160 turns          -> stance_turns.parquet


## Verification — the corpus against the dev bands

Dev bands (0.3-dev, 30 oversampled-long turns): structural-error rate ~3%,
lint a few percent, both stances present with concern dominant. The
corpus-level numbers decide the gate; the per-year table is the first look
at the panel the paper will build on.

In [3]:
err_rate = (turns["errors"] != "").mean()
per_turn = turns["n_records"].mean()
stances = records["stance"].value_counts()
lint = records["topic_lint"].notna().mean()
pool = records["topic"].str.casefold().str.strip().nunique()

assert set(records["stance"]) <= set(stance.STANCES)
assert (turns["prompt_version"] == "stance-extraction-0.3-dev").all()

print(f"1) turns with any structural error: {err_rate:.2%} "
      f"[{'OK' if err_rate < 0.05 else 'OUT OF BAND — inspect'}]")
print(f"2) records/turn : {per_turn:.2f} "
      f"[{'OK' if 0.1 <= per_turn <= 3.0 else 'OUT OF BAND — inspect'}]")
print(f"3) stance mix   : {stances.to_dict()} "
      f"(concern share {stances.get('concern', 0) / len(records):.0%})")
print(f"4) topic lint   : {lint:.2%} flagged — review queue, not removal")
print(f"5) topic pool   : {pool:,} unique phrases (casefold) "
      f"— joins Track-1's 9,095 for taxonomy induction")
print(f"6) empty turns  : {(turns['n_records'] == 0).mean():.0%} "
      f"returned no record (default-to-exclude is a valid outcome)")
print(f"7) merges       : {turns['n_merged'].sum():,} adjacent duplicates "
      f"collapsed at extraction time")

err_counts = (turns.loc[turns["errors"] != "", "errors"].str.split(";")
              .explode().value_counts())
print("\nerror breakdown:")
print(err_counts.to_string())
print("\nrecords by year (chamber mix):")
byyear = records.groupby("year").size()
print(" ".join(f"{y}:{c:,}" for y, c in byyear.items()))

1) turns with any structural error: 1.87% [OK]
2) records/turn : 0.15 [OK]
3) stance mix   : {'concern': 16236, 'support': 16149} (concern share 50%)
4) topic lint   : 2.59% flagged — review queue, not removal
5) topic pool   : 24,798 unique phrases (casefold) — joins Track-1's 9,095 for taxonomy induction
6) empty turns  : 94% returned no record (default-to-exclude is a valid outcome)
7) merges       : 1,815 adjacent duplicates collapsed at extraction time

error breakdown:
errors
indices_not_consecutive    4680
index_out_of_range           21
hit_token_cap                18
json_parse                   17
bad_stance                    9
record_not_object             1

records by year (chamber mix):
1996:8 1999:442 2000:740 2001:1,117 2002:1,259 2003:1,578 2004:821 2005:1,150 2006:1,586 2007:1,851 2008:892 2009:1,249 2010:1,390 2011:1,182 2012:1,268 2013:1,312 2014:1,012 2015:1,468 2016:997 2017:1,913 2018:1,192 2019:1,514 2020:952 2021:1,623 2022:847 2023:1,049 2024:647 2025:1,326
